<a href="https://colab.research.google.com/github/rc2308/churn-ensemble-engine/blob/main/notebooks/06_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — Dual-Mode Dashboard Launcher

**Goal:** Launch the Streamlit dashboard from Colab and expose it via a public URL.

**Modes:**
- Predict mode (no labels) → churn predictions + personas
- Evaluate mode (with `Churn` column) → also shows Gini/AUC


In [19]:
import os
os.chdir("/content")
!rm -rf churn-ensemble-engine
!git clone https://github.com/rc2308/churn-ensemble-engine.git
%cd churn-ensemble-engine
!pip install -q -r requirements.txt
!pip install -q pyngrok


Cloning into 'churn-ensemble-engine'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 80 (delta 34), reused 45 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 1.10 MiB | 14.13 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/churn-ensemble-engine


In [13]:
from google.colab import files
print("Upload your artifacts.zip (from Notebook 04):")
uploaded = files.upload()   # select artifacts.zip


Upload your artifacts.zip (from Notebook 04):


Saving artifacts.zip to artifacts.zip


In [14]:
# Unzip artifacts into the repo
!unzip -o artifacts.zip -d .

# Verify all artifacts are present
import os
print("Artifacts found:")
print(os.listdir("artifacts"))


Archive:  artifacts.zip
 extracting: ./artifacts/ensemble_weight.json  
  inflating: ./artifacts/clust_feats.json  
  inflating: ./artifacts/lgbm_model.joblib  
  inflating: ./artifacts/scaler.joblib  
  inflating: ./artifacts/feature_columns.json  
 extracting: ./artifacts/.gitkeep    
  inflating: ./artifacts/xgb_model.joblib  
  inflating: ./artifacts/persona_map.json  
  inflating: ./artifacts/kmeans.joblib  
Artifacts found:
['xgb_model.joblib', 'kmeans.joblib', '.gitkeep', 'lgbm_model.joblib', 'clust_feats.json', 'scaler.joblib', 'ensemble_weight.json', 'persona_map.json', 'feature_columns.json']


In [24]:
from pyngrok import ngrok, conf

# Paste YOUR ngrok authtoken here
conf.get_default().auth_token = "3DdG4FifQ9iZH0ZyhgFWBBgzoyH_2b2d8DofMc2osfuMAmzP5"
print("ngrok token set ")


ngrok token set 


In [25]:
import subprocess, time, os
from pyngrok import ngrok

ngrok.kill()

# GUARANTEE src is findable via PYTHONPATH
env = os.environ.copy()
env["PYTHONPATH"] = "/content/churn-ensemble-engine"

subprocess.Popen(
    ["streamlit", "run", "app/app.py",
     "--server.port", "8501", "--server.headless", "true"],
    env=env
)

time.sleep(10)
public_url = ngrok.connect(8501)
print("👉 Dashboard URL:", public_url)




👉 Dashboard URL: NgrokTunnel: "https://gutter-motto-bullpen.ngrok-free.dev" -> "http://localhost:8501"


In [26]:
# Upload BankChurners.csv to create a test sample
from google.colab import files
print("Upload BankChurners.csv to create test samples:")
uploaded = files.upload()   # select BankChurners.csv


Upload BankChurners.csv to create test samples:


Saving BankChurners.csv to BankChurners.csv


In [27]:
import pandas as pd

df = pd.read_csv("BankChurners.csv")

# Drop the leakage columns (same as training)
leakage = [c for c in df.columns if "Naive_Bayes" in c]
df = df.drop(columns=leakage)

# Sample 1: WITHOUT label (predict mode) - drop Attrition_Flag
sample_predict = df.drop(columns=["Attrition_Flag"]).head(50)
sample_predict.to_csv("test_predict.csv", index=False)

# Sample 2: WITH label (evaluate mode) - keep Attrition_Flag
sample_evaluate = df.head(50)
sample_evaluate.to_csv("test_evaluate.csv", index=False)

# Download both to test in the dashboard
files.download("test_predict.csv")
files.download("test_evaluate.csv")

print("Test files created and downloading ")
print("- test_predict.csv  → predict mode (no labels)")
print("- test_evaluate.csv → evaluate mode (with labels → Gini shown)")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Test files created and downloading 
- test_predict.csv  → predict mode (no labels)
- test_evaluate.csv → evaluate mode (with labels → Gini shown)
